In [2]:
import pandas as pd 
import numpy as np 
import re 

import matplotlib.pyplot as plt
import seaborn as sns 

import parquet
import os
import gc

In [498]:
stations_vocab = pd.read_parquet('stations_vocab.parquet')
stations_actual_names = pd.read_parquet('stations_actual_names.parquet')


In [499]:
set(stations_vocab.station_id).difference(set(stations_actual_names.station_id))

{7683, 7697, 8272, 8274}

In [3]:
def normalize_station_name(name):
    if pd.isna(name):
        return name
    s = str(name).strip()
    s = re.sub(r"[’']", "", s)      
    s = re.sub(r"\.", "", s)             
    s = re.sub(r"\s+", " ", s)           
    return s

def standardize_intersection(name):
    if pd.isna(name):
        return name
    
    name = str(name).strip()
    
    # if name contains /
    if '/' in name:
        # split by /, strip spaces, sort alfabeticaly, rejoin
        parts = sorted([p.strip().replace(r"\s+", " ") for p in name.split('/')])
        return ' / '.join(parts)
    return name

# df17['start_station_name'] = [re.sub(r'\s*[-–]\s*SMART\b', '', name, flags=re.IGNORECASE) for name in df17.start_station_name]


### 2026

In [4]:
df26 = pd.read_csv('raw_data/bikeshare-ridership-2026.csv')

In [5]:
df26.info()

<class 'pandas.DataFrame'>
RangeIndex: 6830174 entries, 0 to 6830173
Data columns (total 11 columns):
 #   Column              Dtype  
---  ------              -----  
 0   Trip_Id             int64  
 1   Trip_Duration       int64  
 2   Start_Station_Id    int64  
 3   Start_Time          str    
 4   Start_Station_Name  str    
 5   End_Station_Id      float64
 6   End_Time            str    
 7   End_Station_Name    str    
 8   Bike_Id             int64  
 9   User_Type           str    
 10  Bike_Model          str    
dtypes: float64(1), int64(4), str(6)
memory usage: 1.2 GB


In [6]:
df26.isna().sum() 

Trip_Id                   0
Trip_Duration             0
Start_Station_Id          0
Start_Time                0
Start_Station_Name        0
End_Station_Id        12951
End_Time               8065
End_Station_Name      12951
Bike_Id                   0
User_Type                 0
Bike_Model                0
dtype: int64

In [ ]:
# End_Station_Id == End_Station_Name, can't recover from vocabulary, 
# End_Time - can recover potentially from Start_Time + Trip_Duration

In [9]:
df26.columns = df26.columns.str.lower().str.replace(r"\s+", "_", regex=True)

In [504]:
df26['start_time'] = pd.to_datetime(df26['start_time'])
df26['end_time'] = pd.to_datetime(df26['end_time'])

In [ ]:
df26.memory_usage(deep=True).sum() / 1e9


np.float64(1.032124101)

In [12]:
df26.bike_model.nunique()
df26.bike_model.unique()

<ArrowStringArray>
['ICONIC', 'EFIT', 'ASTRO']
Length: 3, dtype: str

In [ ]:
# bike_model_map = {'ICONIC': 0, 'EFIT': 1, 'ASTRO': 2, 'Unknown': -1}
# df26['bike_model'] = df26['bike_model'].map(bike_model_map)

In [13]:
df26['user_type'] = df26['user_type'].astype('category')
df26['bike_model'] = df26['bike_model'].astype('category')

In [14]:
# categorizing station_name columns
df26['start_station_name'] = df26['start_station_name'].astype('category')
df26['end_station_name'] = df26['end_station_name'].astype('category')
df26.memory_usage(deep=True).sum() / 1e9


np.float64(0.423546272)

In [506]:
# downcasting numeric columns
int_cols = df26.select_dtypes(include='int64').columns
df26[int_cols] = df26[int_cols].apply(pd.to_numeric, downcast='integer')

float_cols = df26.select_dtypes(include='float64').columns
df26[float_cols] = df26[float_cols].apply(pd.to_numeric, downcast='float')
df26.memory_usage(deep=True).sum() / 1e9

np.float64(0.868199925)

In [204]:
df26.start_time.dt.year.unique()

array([2026], dtype=int32)

In [ ]:
#result - from 1.292371177 GB to 0.259622096 GB

In [208]:
# sations names normalization:
df26['start_station_name'] = [normalize_station_name(name) for name in df26.start_station_name]
df26['end_station_name'] = [normalize_station_name(name) for name in df26.end_station_name]

In [507]:
#put all steps into function 
def optimize_df(df):
    if 'start_time' in df.columns:
        df['start_time'] = pd.to_datetime(df['start_time'])
    if 'end_time' in df.columns:
        df['end_time'] = pd.to_datetime(df['end_time'])

    for col in ['bike_model', 'user_type', 'start_station_name', 'end_station_name']:
        if col in df.columns:
            df[col] = df[col].astype('category')

    for col in ['trip_id', 'trip_duration', 'start_station_id', 'end_station_id', 'bike_id']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    int_cols = df.select_dtypes(include='int64').columns
    df[int_cols] = df[int_cols].apply(pd.to_numeric, downcast='integer')
    float_cols = df.select_dtypes(include='float64').columns
    df[float_cols] = df[float_cols].apply(pd.to_numeric, downcast='float')
    return df

In [508]:
# 
df26 = optimize_df(df26)

In [211]:
df26.memory_usage(deep=True).sum() / 1e9

np.float64(0.259621944)

In [216]:
df26.to_parquet('clean_years/df26.parquet', index=False)


In [10]:
st26 = df26[['start_station_id', 'start_station_name']].drop_duplicates()

In [11]:
del df26
gc.collect()

116

### 2025

In [12]:
path25 = 'raw_data/bikeshare-ridership-2025'


In [13]:
files25 = os.listdir(path25)

monthly_dfs_25 = [
   pd.read_csv(os.path.join(path25, f), encoding='cp1252') for f in files25
]

In [14]:
df25 = pd.concat(monthly_dfs_25, ignore_index=True)
df25.head()

,Trip_Id,Trip_Duration,Start_Station_Id,Start_Time,Start_Station_Name,End_Station_Id,End_Time,End_Station_Name,Bike_Id,User_Type,Bike_Model
0,41865752,347,7389.0,2025-10-01 00:01:03,College Park - Gerrard Entrance,7693.0,2025-10-01 00:06:50,College Park - Gerrard Entrance,5579,Member,ICONIC
1,41865762,806,7485.0,2025-10-01 00:02:49,Blythwood Rd / Yonge St - SMART,7676.0,2025-10-01 00:16:15,Blythwood Rd / Yonge St - SMART,6681,Casual,EFIT
2,41865748,321,7030.0,2025-10-01 00:00:32,Bay St / Wellesley St W,7058.0,2025-10-01 00:05:53,Bay St / Wellesley St W,1392,Casual,ICONIC
3,41865749,125,7386.0,2025-10-01 00:00:48,D'Arcy St / McCaul St - SMART,7257.0,2025-10-01 00:02:53,D'Arcy St / McCaul St - SMART,2545,Member,ICONIC
4,41865767,782,7894.0,2025-10-01 00:03:23,Millwood Rd / Southvale Dr,7194.0,2025-10-01 00:16:25,Millwood Rd / Southvale Dr,9947,Member,EFIT


In [10]:
df25.info()

<class 'pandas.DataFrame'>
RangeIndex: 7812520 entries, 0 to 7812519
Data columns (total 11 columns):
 #   Column              Dtype  
---  ------              -----  
 0   Trip_Id             int64  
 1   Trip_Duration       int64  
 2   Start_Station_Id    float64
 3   Start_Time          str    
 4   Start_Station_Name  str    
 5   End_Station_Id      float64
 6   End_Time            str    
 7   End_Station_Name    str    
 8   Bike_Id             int64  
 9   User_Type           str    
 10  Bike_Model          str    
dtypes: float64(2), int64(3), str(6)
memory usage: 1.4 GB


In [13]:
df25.isna().sum()

trip_id                  0
trip_duration            0
start_station_id         7
start_time               0
start_station_name       7
end_station_id        6761
end_time              1092
end_station_name       924
bike_id                  0
user_type                0
bike_model               0
dtype: int64

In [ ]:
# i can recover some end_station_id from(missing rows 6761) end_station_name (missing rows 924)
# also end_time  =  start_time + trip_duration

In [15]:
df25.columns = df25.columns.str.lower().str.replace(r"\s+", "_", regex=True)

In [16]:
st25 = df25[['start_station_id', 'start_station_name']]

In [ ]:
# normalize station names 
df25['start_station_name'] = [normalize_station_name(name) for name in df25.start_station_name]
df25['end_station_name'] = [normalize_station_name(name) for name in df25.end_station_name]

#### Information recovery for end_station_id 2025

In [18]:
stations_vocab.columns

Index(['station_id', 'name'], dtype='str')

In [36]:
df25 = (df25.merge(stations_vocab, 
            left_on = 'end_station_name', 
            right_on = 'name', 
            how='left')
    .drop(['end_station_id', 'name'], axis=1)
    .rename(columns = {'station_id': 'end_station_id'})
#     .isna().sum() missings for end_station_id = end_station_name = 924 obs
    )

In [37]:
# optimize df25
print("before: ", df25.memory_usage(deep=True).sum() / 1e9)
df25 = optimize_df(df25)
print("after: ", df25.memory_usage(deep=True).sum()/ 1e9)

before:  1.558500022
after:  0.328149302


In [ ]:
#from 1.484543626 GB to  0.31257724 GB for 2025

In [38]:
df25.start_time.dt.year.unique()

array([2025], dtype=int32)

In [39]:
df25.info()

<class 'pandas.DataFrame'>
RangeIndex: 8201916 entries, 0 to 8201915
Data columns (total 11 columns):
 #   Column              Dtype         
---  ------              -----         
 0   trip_id             int32         
 1   trip_duration       int32         
 2   start_station_id    float32       
 3   start_time          datetime64[us]
 4   start_station_name  category      
 5   end_time            datetime64[us]
 6   end_station_name    category      
 7   bike_id             int16         
 8   user_type           category      
 9   bike_model          category      
 10  end_station_id      float32       
dtypes: category(4), datetime64[us](2), float32(2), int16(1), int32(2)
memory usage: 312.9 MB


In [52]:
del monthly_dfs_25
gc.collect()

74

In [ ]:
df25.to_parquet('clean_years/df25.parquet', index=False)

In [20]:
del monthly_dfs_25

In [ ]:
del df25
gc.collect()

### 2024

In [22]:
df24 = pd.read_csv('raw_data/bikeshare-ridership-2024.csv')

In [23]:
df24.info()

<class 'pandas.DataFrame'>
RangeIndex: 6953094 entries, 0 to 6953093
Data columns (total 11 columns):
 #   Column              Dtype  
---  ------              -----  
 0   Trip_Id             int64  
 1   Trip_Duration       int64  
 2   Start_Station_Id    int64  
 3   Start_Time          str    
 4   Start_Station_Name  str    
 5   End_Station_Id      float64
 6   End_Time            str    
 7   End_Station_Name    str    
 8   Bike_Id             int64  
 9   User_Type           str    
 10  Bike_Model          str    
dtypes: float64(1), int64(4), str(6)
memory usage: 1.2 GB


In [44]:
df24.isna().sum()

Trip_Id                  0
Trip_Duration            0
Start_Station_Id         0
Start_Time               0
Start_Station_Name       0
End_Station_Id        8424
End_Time              5822
End_Station_Name      8424
Bike_Id                  0
User_Type                0
Bike_Model               0
dtype: int64

In [ ]:
# nothing to recover missings for End_Station_Name = End_Station_Id
# can be done for specific analysis in the future

In [24]:
df24.columns = [c.lower().replace(' ', '_') for c in df24.columns]
print(df24.shape[1])
df24.columns

11


Index(['trip_id', 'trip_duration', 'start_station_id', 'start_time',
       'start_station_name', 'end_station_id', 'end_time', 'end_station_name',
       'bike_id', 'user_type', 'bike_model'],
      dtype='str')

In [25]:
st24 = df24[['start_station_id', 'start_station_name']]

In [48]:
# normalize station names 
df24['start_station_name'] = [normalize_station_name(name) for name in df24.start_station_name]
df24['end_station_name'] = [normalize_station_name(name) for name in df24.end_station_name]

In [49]:
# optimize df24 for memory:
print(df24.memory_usage(deep=True).sum() / 1e9)
df24 = optimize_df(df24)
print(df24.memory_usage(deep=True).sum() / 1e9)

1.323263256
0.264278046


In [51]:
df24.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 6953094 entries, 0 to 6953093
Data columns (total 11 columns):
 #   Column              Dtype         
---  ------              -----         
 0   trip_id             int32         
 1   trip_duration       int32         
 2   start_station_id    int16         
 3   start_time          datetime64[us]
 4   start_station_name  category      
 5   end_station_id      float32       
 6   end_time            datetime64[us]
 7   end_station_name    category      
 8   bike_id             int16         
 9   user_type           category      
 10  bike_model          category      
dtypes: category(4), datetime64[us](2), float32(1), int16(2), int32(2)
memory usage: 252.0 MB


In [ ]:
# 1.324427269 -> 0.264278176 GB

In [53]:
df24.to_parquet('clean_years/df24.parquet')

In [26]:
del df24
gc.collect()

8

### 2023

In [27]:
path23 = 'raw_data/bikeshare-ridership-2023'
os.listdir(path23)

['Bike share ridership 2023-08.csv',
 'Bike share ridership 2023-09.csv',
 'Bike share ridership 2023-02.csv',
 'Bike share ridership 2023-03.csv',
 'Bike share ridership 2023-01.csv',
 'Bike share ridership 2023-10.csv',
 'Bike share ridership 2023-04.csv',
 'Bike share ridership 2023-05.csv',
 'Bike share ridership 2023-11.csv',
 'Bike share ridership 2023-07.csv',
 'Bike share ridership 2023-12.csv',
 'Bike share ridership 2023-06.csv']

In [28]:
files23 = os.listdir(path23)

monthly_dfs_23 = [
   pd.read_csv(os.path.join(path23, f), encoding='cp1252') for f in files23
]

In [29]:
df23 = pd.concat(monthly_dfs_23, ignore_index=True)
df23.head()

,Trip Id,Trip Duration,Start Station Id,Start Time,Start Station Name,End Station Id,End Time,End Station Name,Bike Id,User Type,ï»¿Trip Id
0,23529880.0,145,7101,08/01/2023 00:00,Lower Sherbourne St / The Esplanade,7291.0,08/01/2023 00:02,190 Queens Quay E,976,Casual Member,NaN
1,23529881.0,473,7040,08/01/2023 00:00,Euclid Ave / Bloor St W,7180.0,08/01/2023 00:08,Lansdowne Subway Station,1827,Casual Member,NaN
2,23529882.0,394,7787,08/01/2023 00:00,NaN,7067.0,08/01/2023 00:07,Yonge St / Harbour St,67,Casual Member,NaN
3,23529883.0,458,7048,08/01/2023 00:00,Front St W / Yonge St (Hockey Hall of Fame),7077.0,08/01/2023 00:08,College Park South,2293,Casual Member,NaN
4,23529884.0,654,7387,08/01/2023 00:01,Mortimer Ave / Carlaw Ave SMART,7338.0,08/01/2023 00:11,Logan Ave / Bain Ave,7402,Casual Member,NaN


In [122]:
df23.shape

(5713141, 11)

In [121]:
df23.isna().sum()

Trip Id                834571
Trip  Duration              0
Start Station Id            0
Start Time                  0
Start Station Name     595075
End Station Id           2944
End Time                    0
End Station Name       598563
Bike Id                     0
User Type                   0
ï»¿Trip Id            4878570
dtype: int64

In [ ]:
834571+4878570
# it's turned out, that we have trip ids in 2 columns

5713141

In [529]:
df23=df23.rename(columns={df23.columns[-1]: 'trip_id2'})

In [530]:
df23.columns

Index(['Trip Id', 'Trip  Duration', 'Start Station Id', 'Start Time',
       'Start Station Name', 'End Station Id', 'End Time', 'End Station Name',
       'Bike Id', 'User Type', 'trip_id2'],
      dtype='str')

In [30]:
df23.columns = df23.columns.str.lower().str.replace(r"\s+", "_", regex=True)

print(df23.shape[1])
df23.columns

11


Index(['trip_id', 'trip_duration', 'start_station_id', 'start_time',
       'start_station_name', 'end_station_id', 'end_time', 'end_station_name',
       'bike_id', 'user_type', 'ï»¿trip_id'],
      dtype='str')

In [532]:
df23['trip_id'] = df23['trip_id'].combine_first(df23['trip_id2'])

In [533]:
df23 = df23.iloc[:, :-1]

In [31]:
st23 = df23[['start_station_id', 'start_station_name']]

In [148]:
df23.isna().sum()

trip_id                    0
trip_duration              0
start_station_id           0
start_time                 0
start_station_name    595075
end_station_id          2944
end_time                   0
end_station_name      598563
bike_id                    0
user_type                  0
dtype: int64

In [149]:
df23['start_station_name'] = [normalize_station_name(name) for name in df23.start_station_name]
df23['end_station_name'] = [normalize_station_name(name) for name in df23.end_station_name]

In [ ]:
# I can recover start_station_name (595075 miss) from start_station_id (0 missings)
# and end_station_name (598563 missings) from end_station_id (end_station_id missings )

#### Information recovery for start_station_name and end_station_name 2023

In [150]:
stations_actual_names.columns

Index(['station_id', 'name'], dtype='str')

In [151]:
df23 = (df23.merge(stations_actual_names, 
           left_on='start_station_id', 
           right_on = 'station_id', 
           how = 'left')
        .drop(['start_station_name', 'station_id'], axis=1)
        .rename(columns={'name': 'start_station_name'})
 )


(df23
    .query('start_station_name.isna()')['start_station_id']
    .drop_duplicates())



1515513    7683
3089192    7690
Name: start_station_id, dtype: int64

In [152]:
stations_vocab.loc[stations_vocab.station_id.isin([7683, 7690])]

,station_id,name
1091,7683,Bay St / Harbour St (West)


In [153]:
# let's check also wider stations_vocab

df23 = (df23.merge(stations_vocab.loc[stations_vocab.station_id.isin([7683, 7690])], 
            how='left', 
            left_on='start_station_id', 
            right_on = 'station_id')
    .assign(start_station_name = lambda d: d['start_station_name'].combine_first(d['name']))
    .drop(['name', 'station_id'], axis=1)
 )

In [ ]:
df23.isna().sum()
# now start_station_name misses just for 1 trip, let's check whether it has valuable information

trip_id                    0
trip_duration              0
start_station_id           0
start_time                 0
end_station_id          2944
end_time                   0
end_station_name      598563
bike_id                    0
user_type                  0
start_station_name         1
dtype: int64

In [155]:
df23[df23.start_station_name.isna()]

,trip_id,trip_duration,start_station_id,start_time,end_station_id,end_time,end_station_name,bike_id,user_type,start_station_name
3089192,21268121.0,0,7690,05/02/2023 12:07,7175.0,05/02/2023 12:07,HTO Park (Queens Quay W),5269,Casual Member,NaN


In [ ]:
# trip_duration = 0, start_time = end_time, 
# for my analysis it's not valuable trip, where i take into consideretion just "successful" trips.
# for now definition of successful trip, it's where trip_duration>0 (or 30 sec/1 min?); start_time = end_time; 
# potentially start_station_id = end_station_id

In [ ]:
# drop a row with id = 7690
df23 = df23[~df23.start_station_name.isna()]

In [158]:
# optimize df23 for memory usage:
# optimize df24 for memory:
print(df23.memory_usage(deep=True).sum() / 1e9)
df23 = optimize_df(df23)


1.048720604


In [162]:
df23.info(memory_usage = 'deep')

<class 'pandas.DataFrame'>
Index: 5713140 entries, 0 to 5713140
Data columns (total 10 columns):
 #   Column              Dtype         
---  ------              -----         
 0   trip_id             float64       
 1   trip_duration       int32         
 2   start_station_id    int16         
 3   start_time          datetime64[us]
 4   end_station_id      float32       
 5   end_time            datetime64[us]
 6   end_station_name    category      
 7   bike_id             int16         
 8   user_type           category      
 9   start_station_name  category      
dtypes: category(3), datetime64[us](2), float32(1), float64(1), int16(2), int32(1)
memory usage: 267.0 MB


In [163]:
df23.to_parquet('clean_years/df23.parquet')

In [32]:
del monthly_dfs_23, df23
gc.collect()

6

### 2022

In [33]:
path22 = 'raw_data/bikeshare-ridership-2022'
files22 = os.listdir(path22)


In [34]:
monthly_dfs_22 = [pd.read_csv(os.path.join(path22, f), encoding='cp1252') for f in os.listdir(path22)]

In [35]:
df22 = pd.concat(monthly_dfs_22, ignore_index=True)
df22.head()

,ï»¿Trip Id,Trip Duration,Start Station Id,Start Time,Start Station Name,End Station Id,End Time,End Station Name,Bike Id,User Type,Trip Id
0,14805109.0,4335,7334,01/01/2022 00:02,Simcoe St / Wellington St North,7269.0,01/01/2022 01:15,Toronto Eaton Centre (Yonge St),5139,Casual Member,NaN
1,14805110.0,126,7443,01/01/2022 00:02,Dundas St E / George St,7270.0,01/01/2022 00:05,Church St / Dundas St E - SMART,3992,Annual Member,NaN
2,14805112.0,942,7399,01/01/2022 00:04,Lower Jarvis / Queens Quay E,7686.0,01/01/2022 00:19,NaN,361,Annual Member,NaN
3,14805113.0,4256,7334,01/01/2022 00:04,Simcoe St / Wellington St North,7269.0,01/01/2022 01:15,Toronto Eaton Centre (Yonge St),4350,Casual Member,NaN
4,14805114.0,4353,7334,01/01/2022 00:05,Simcoe St / Wellington St North,7038.0,01/01/2022 01:17,Dundas St W / Yonge St,5074,Casual Member,NaN


In [539]:
df22 = df22.rename(columns = {df22.columns[0]: 'trip_id2'})
df22.columns

Index(['trip_id2', 'Trip  Duration', 'Start Station Id', 'Start Time',
       'Start Station Name', 'End Station Id', 'End Time', 'End Station Name',
       'Bike Id', 'User Type', 'Trip Id'],
      dtype='str')

In [36]:
df22.columns = df22.columns.str.lower().str.replace(r"\s+", "_", regex=True)

In [541]:
df22['trip_id'] = df22['trip_id'].combine_first(df22['trip_id2'])

In [542]:
df22 = df22.drop('trip_id2', axis=1)

In [315]:
df22.isna().sum()

# start_station_name (183466 missings), start_station_id (0 missings)
# end_station_name (184966 missings), end_station_id (1279 missings)

trip_duration              0
start_station_id           0
start_time                 0
start_station_name    183466
end_station_id          1279
end_time                   0
end_station_name      184966
bike_id                    0
user_type                  0
trip_id                    0
dtype: int64

In [37]:
st22 = df22[['start_station_id', 'start_station_name']]

In [316]:
df22['start_station_name'] = [normalize_station_name(name) for name in df22.start_station_name]
df22['end_station_name'] = [normalize_station_name(name) for name in df22.end_station_name]

### Information recovery for start_station_name/end_station_name 2022

In [317]:
df22 = (df22.merge(stations_actual_names, 
           left_on='start_station_id', 
           right_on = 'station_id', 
           how = 'left')
        .drop(['start_station_name', 'station_id'], axis=1)
        .rename(columns={'name': 'start_station_name'})
 )



In [318]:
missed_start_ids = (df22
    .query('start_station_name.isna()')['start_station_id']
    .drop_duplicates().tolist())

In [319]:
stations_vocab[stations_vocab.station_id.isin(missed_start_ids)]

,station_id,name
1091,7683,Bay St / Harbour St (West)
1116,7697,St Clair Ave W / Cloverdale Rd - SMART Vault


In [320]:
# let's check in statinos_vocab

df22 = (df22.merge(stations_vocab[stations_vocab.station_id.isin(missed_start_ids)], 
           left_on='start_station_id', 
           right_on = 'station_id', 
           how = 'left')
 )

In [321]:
df22['start_station_name'] = df22['start_station_name'].combine_first(df22['name'])


In [322]:
df22 = df22.drop(['station_id', 'name'], axis=1)
df22.isna().sum()

trip_duration              0
start_station_id           0
start_time                 0
end_station_id          1279
end_time                   0
end_station_name      184966
bike_id                    0
user_type                  0
trip_id                    0
start_station_name       627
dtype: int64

In [323]:
df22[df22.start_station_name.isna() & (df22.trip_duration==0)]
# all trips without name look valid.

,trip_duration,start_station_id,start_time,end_station_id,end_time,end_station_name,bike_id,user_type,trip_id,start_station_name
4328527,0,7714,08/18/2022 23:05,7132.0,08/18/2022 23:05,Kendal Ave / Spadina Rd,6523,Casual Member,17986316.0,NaN


In [324]:
unmatched_ids = df22[df22.start_station_name.isna()]['start_station_id'].unique()


In [325]:
unmatched_ids = list(unmatched_ids)

In [326]:
unmatched_ids

[np.int64(7714), np.int64(7722)]

In [38]:
del df22, monthly_dfs_22
gc.collect()

6

### 2021

In [ ]:
from charset_normalizer import from_path

def detect_encoding(path, sample_size=100_000):
    result = from_path(path).best()
    return result.encoding if result else 'utf-8'


def read_csv_robust(path, **kwargs):
    # utf-8-sig strips a leading BOM  (UTF-8 BOM -  Byte Order Mark)
    for enc in ['utf-8-sig', detect_encoding(path), 'cp1252', 'latin-1']:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except (UnicodeDecodeError, LookupError):
            continue
    raise ValueError(f"Could not decode {path} with any encoding")

In [143]:
# import io

# def read_lines_robust(path, encodings=('utf-8-sig', 'cp1252', 'latin-1')):
#     with open(path, 'rb') as f:
#         raw_lines = f.readlines()
#     decoded = []
#     for raw in raw_lines:
#         for enc in encodings:
#             try:
#                 decoded.append(raw.decode(enc))
#                 break
#             except UnicodeDecodeError:
#                 continue
#     return decoded

# def read_csv_robust(path):
#     lines = read_lines_robust(path)
#     return pd.read_csv(io.StringIO(''.join(lines)))

    


In [132]:
path21 = 'raw_data/bikeshare-ridership-2021'
files21 = os.listdir(path21)

In [139]:
monthly_dfs_21 = [read_csv_robust(os.path.join(path21, f)) for f in files21]

In [ ]:
# monthly_dfs_21 = [pd.read_csv(os.path.join(path21, f), encoding='cp1252') for f in files21]

In [140]:
df21 = pd.concat(monthly_dfs_21, ignore_index=True)
df21.head()

,Trip Id,Trip Duration,Start Station Id,Start Time,Start Station Name,End Station Id,End Time,End Station Name,Bike Id,User Type
0,12823998,1347,7018,08/01/2021 00:00,Bremner Blvd / Rees St,7018.0,08/01/2021 00:22,Bremner Blvd / Rees St,35.0,Casual Member
1,12823999,1348,7018,08/01/2021 00:00,Bremner Blvd / Rees St,7018.0,08/01/2021 00:22,Bremner Blvd / Rees St,222.0,Casual Member
2,12824000,658,7102,08/01/2021 00:00,Nelson St / Duncan St,7077.0,08/01/2021 00:11,College Park South,2684.0,Casual Member
3,12824002,639,7102,08/01/2021 00:00,Nelson St / Duncan St,7077.0,08/01/2021 00:11,College Park South,520.0,Casual Member
4,12824003,1070,7030,08/01/2021 00:00,Bay St / Wellesley St W,7202.0,08/01/2021 00:18,Queen St W / York St (City Hall),3500.0,Casual Member


In [141]:
df21[df21['Start Station Id']==7070]['Start Station Name'].unique()

<ArrowStringArray>
['25 York St GÇô Union Station South', '25 York St – Union Station South']
Length: 2, dtype: str

In [547]:
df21 = df21.rename(columns = {df21.columns[0]: 'trip_id2'})

In [142]:
df21.columns = df21.columns.str.lower().str.replace(r"\s+", "_", regex=True)
df21.columns

Index(['trip_id', 'trip_duration', 'start_station_id', 'start_time',
       'start_station_name', 'end_station_id', 'end_time', 'end_station_name',
       'bike_id', 'user_type'],
      dtype='str')

In [43]:
st21 = df21[['start_station_id', 'start_station_name']]

In [231]:
df21['trip_id'] = df21['trip_id'].combine_first(df21['trip_id2'])
df21 = df21.drop('trip_id2', axis=1)

In [232]:
df21['start_station_name'] = [normalize_station_name(name) for name in df21.start_station_name]
df21['end_station_name'] = [normalize_station_name(name) for name in df21.end_station_name]

In [233]:
df21.isna().sum()

trip_duration            0
start_station_id         0
start_time               0
start_station_name    3680
end_station_id        1969
end_time                 0
end_station_name      5894
bike_id                203
user_type                0
trip_id                  0
dtype: int64

In [ ]:
# try restore start_station_name from start_station_id

### Information recovery for start_station_name 2021

In [ ]:
df21 = df21.merge(stations_actual_names,
           how='left',
           left_on = 'start_station_id',
           right_on = 'station_id')

df21['start_station_name'] = df21['start_station_name'].combine_first(df21['name'])

In [236]:
df21 = df21.drop(['name', 'station_id'], axis=1)

In [ ]:
df21.isna().sum() # still 467 unmatched trips. let's check how many unmatched ids

trip_duration            0
start_station_id         0
start_time               0
start_station_name     467
end_station_id        1969
end_time                 0
end_station_name      5894
bike_id                203
user_type                0
trip_id                  0
dtype: int64

In [255]:
df21[df21.start_station_name.isna()]['start_station_id'].unique()

array([7683])

In [257]:
# let's check in full vocab
stations_vocab[stations_vocab.station_id==7683]

,station_id,name
1091,7683,Bay St / Harbour St (West)


In [258]:
# let's check in full vocab
df21 = (df21.merge(stations_vocab[stations_vocab.station_id==7683],
           how='left', 
           left_on = 'start_station_id', 
           right_on = 'station_id')
    .assign(start_station_name = lambda d: d['start_station_name'].combine_first(d['name']))
    .drop(['station_id', 'name'], axis=1)
    )

In [ ]:
df21.isna().sum() # start_station_namefully recoevered

trip_duration            0
start_station_id         0
start_time               0
start_station_name       0
end_station_id        1969
end_time                 0
end_station_name      5894
bike_id                203
user_type                0
trip_id                  0
dtype: int64

In [ ]:
# unmatched_ids = list(unmatched_ids)
# unmatched_ids.extend( df21[df21.start_station_name.isna()]['start_station_id'].unique())

# del unmatched_ids[2]

In [263]:
# memory optimization:
print('before:', df21.memory_usage(deep=True).sum()/1e9)
df21 = optimize_df(df21)
print('after:', df21.memory_usage(deep=True).sum()/1e9)


before: 0.640904095
after: 0.139485162


In [264]:
df21.to_parquet('clean_years/df21.parquet', index=False)

In [44]:
# free ram
del df21
gc.collect()

6

### 2020

In [45]:
# monthly_dfs_20 = [
#     pd.read_csv(os.path.join('bikeshare-ridership-2020', file)) for file in os.listdir('bikeshare-ridership-2020')
# ]
path20 = 'raw_data/bikeshare-ridership-2020'

In [46]:
# conundrum with 2020-10
# problem with 2020-10.csv, 249 rows are missing the Trip Duration field entirely, 
# which shifts every subsequent column one position to the left for those rows. 
df20_10 = pd.read_csv(os.path.join(path20, '2020-10.csv'))


In [47]:
df20_10 = df20_10.astype('object')

In [48]:
cols = df20_10.columns.tolist()

In [49]:
bad_rows = pd.to_numeric(df20_10[cols[2]], errors='coerce').isna() & df20_10[cols[2]].notna()

# df20_10.loc[bad_rows, cols[2:]].dtypes
df20_10.loc[bad_rows, cols[2:]] = df20_10.loc[bad_rows, cols[1:-1]].values
# df20_10.loc[bad_rows, cols[1]] = np.nan

In [50]:
df20_10.loc[bad_rows, cols[1]] = np.nan

In [51]:
# rest of the 2020 except corrupted 10/20: 

monthly_dfs_20 = [
    pd.read_csv(os.path.join(path20, file)) for file in os.listdir(path20) if file !='2020-10.csv'
]

In [52]:
df20 = pd.concat(monthly_dfs_20, ignore_index=True)
df20.head()

,Trip Id,Trip Duration,Start Station Id,Start Time,Start Station Name,End Station Id,End Time,End Station Name,Bike Id,User Type
0,7452736,625,7162,02/01/2020 00:00,Hayter St / Laplante Ave,7190.0,02/01/2020 00:11,St. George St / Hoskin Ave,653.0,Annual Member
1,7452737,426,7170,02/01/2020 00:01,Spadina Ave / Willcocks St,7239.0,02/01/2020 00:08,Bloor St W / Manning Ave - SMART,4682.0,Annual Member
2,7452738,817,7475,02/01/2020 00:03,Jarvis St / Richmond St E,7075.0,02/01/2020 00:16,Queens Quay W / Dan Leckie Way,3852.0,Annual Member
3,7452739,653,7099,02/01/2020 00:03,Cherry St / Mill St,7292.0,02/01/2020 00:14,Granby St / Church St - SMART,4546.0,Annual Member
4,7452740,189,7176,02/01/2020 00:03,Bathurst St / Fort York Blvd,7402.0,02/01/2020 00:06,Wellington St W / Bathurst St,4867.0,Annual Member


In [53]:
df20.columns = df20.columns.str.lower().str.replace(r"\s+", "_", regex=True)
df20.columns

Index(['trip_id', 'trip_duration', 'start_station_id', 'start_time',
       'start_station_name', 'end_station_id', 'end_time', 'end_station_name',
       'bike_id', 'user_type'],
      dtype='str')

In [283]:
sample_idx = np.random.choice(df20.shape[0], 15)
print(df20['start_time'].iloc[sample_idx].tolist())


['09/04/2020 20:56', '07/09/2020 15:59', '08/19/2020 09:03', '06/12/2020 06:16', '09/07/2020 09:13', '06/25/2020 22:47', '11/09/2020 17:15', '07/22/2020 14:54', '08/31/2020 17:18', '07/18/2020 13:14', '08/05/2020 18:45', '07/03/2020 22:09', '06/14/2020 16:58', '08/07/2020 15:26', '05/14/2020 11:00']


In [285]:
pd.to_datetime(df20['start_time'])[:4]

0   2020-02-01 00:00:00
1   2020-02-01 00:01:00
2   2020-02-01 00:03:00
3   2020-02-01 00:03:00
Name: start_time, dtype: datetime64[us]

In [54]:
df20_10.columns = df20_10.columns.str.lower().str.replace(r"\s+", "_", regex=True)
df20_10.columns

Index(['trip_id', 'trip_duration', 'start_station_id', 'start_time',
       'start_station_name', 'end_station_id', 'end_time', 'end_station_name',
       'bike_id', 'user_type'],
      dtype='str')

In [561]:
pd.to_datetime(df20_10['start_time'])[:4]

0   2020-10-01 00:00:00
1   2020-10-01 00:00:00
2   2020-10-01 00:01:00
3   2020-10-01 00:01:00
Name: start_time, dtype: datetime64[us]

In [55]:
df20 = pd.concat([df20, df20_10])
df20.info()

<class 'pandas.DataFrame'>
Index: 2911308 entries, 0 to 277838
Data columns (total 10 columns):
 #   Column              Dtype 
---  ------              ----- 
 0   trip_id             object
 1   trip_duration       object
 2   start_station_id    object
 3   start_time          object
 4   start_station_name  object
 5   end_station_id      object
 6   end_time            object
 7   end_station_name    object
 8   bike_id             object
 9   user_type           object
dtypes: object(10)
memory usage: 244.3+ MB


In [294]:
df20.isna().sum()

trip_id                  0
trip_duration          249
start_station_id         0
start_time               0
start_station_name     793
end_station_id        1301
end_time                 0
end_station_name      2049
bike_id                 73
user_type                0
dtype: int64

In [56]:
st20 = df20[['start_station_id', 'start_station_name']]

In [295]:
df20['start_station_name'] = [normalize_station_name(name) for name in df20.start_station_name]
df20['end_station_name'] = [normalize_station_name(name) for name in df20.end_station_name]

In [296]:
df20 = (df20.merge(stations_actual_names,
           how='left',
           left_on='start_station_id',
           right_on='station_id'
           )
    .assign(start_station_name = lambda d: d['start_station_name'].combine_first(d['name']))
    .drop(['name', 'station_id'], axis=1)
 )

In [ ]:
df20.isna().sum() # still 164 tips are missed

trip_id                  0
trip_duration          249
start_station_id         0
start_time               0
start_station_name     164
end_station_id        1301
end_time                 0
end_station_name      2049
bike_id                 73
user_type                0
dtype: int64

In [300]:
list(df20[df20.start_station_name.isna()]['start_station_id'].unique())

['7659', '7660']

In [302]:
# lets check them in stations_vocab
stations_vocab[stations_vocab.station_id.isin(list(df20[df20.start_station_name.isna()]['start_station_id'].unique()))]

,station_id,name


In [328]:
# missed in every station tbls, add to unmatched_ids:
unmatched_ids.extend(list(df20[df20.start_station_name.isna()]['start_station_id'].unique()))
unmatched_ids

[np.int64(7714), np.int64(7722), '7659', '7660']

In [329]:
print('before: ', df20.memory_usage(deep=True).sum()/1e9)
df20 = optimize_df(df20)
print('after: ', df20.memory_usage(deep=True).sum()/1e9)

before:  1.283785175
after:  0.12523013


In [330]:
df20.to_parquet('clean_years/df20.parquet', index=False)

In [57]:
del df20, monthly_dfs_20, df20_10
gc.collect()

12

### 2019

In [58]:
path19 = 'raw_data/bikeshare-ridership-2019'

quaterly_dfs_19 = [pd.read_csv(os.path.join(path19, file)) for file in os.listdir(path19)]

In [59]:
df19 = pd.concat(quaterly_dfs_19, ignore_index=True)
df19.head()

,Trip Id,Trip Duration,Start Station Id,Start Time,Start Station Name,End Station Id,End Time,End Station Name,Bike Id,User Type
0,5533843,1834.0,7352,07/01/2019 00:00,Cosburn Ave / Donlands Ave,7352.0,07/01/2019 00:30,Cosburn Ave / Donlands Ave,989,Casual Member
1,5533844,338.0,7016,07/01/2019 00:00,Bay St / Queens Quay W (Ferry Terminal),7068.0,07/01/2019 00:06,Dockside Dr / Queens Quay E (Sugar Beach),4006,Annual Member
2,5533845,1433.0,7193,07/01/2019 00:00,Queen St W / Gladstone Ave,7285.0,07/01/2019 00:24,Spadina Ave / Harbord St - SMART,1512,Casual Member
3,5533846,1042.0,7369,07/01/2019 00:00,Dupont St / Emerson Ave SMART,7239.0,07/01/2019 00:18,Bloor St W / Manning Ave - SMART,3219,Casual Member
4,5533847,4203.0,7168,07/01/2019 00:01,Queens Quay / Yonge St,7226.0,07/01/2019 01:11,Lake Shore Blvd W / The Boulevard Club,4425,Casual Member


In [566]:
df19.info()

<class 'pandas.DataFrame'>
RangeIndex: 2439517 entries, 0 to 2439516
Data columns (total 10 columns):
 #   Column              Dtype  
---  ------              -----  
 0   Trip Id             int64  
 1   Trip  Duration      float64
 2   Start Station Id    int64  
 3   Start Time          str    
 4   Start Station Name  str    
 5   End Station Id      float64
 6   End Time            str    
 7   End Station Name    str    
 8   Bike Id             int64  
 9   User Type           str    
dtypes: float64(2), int64(3), str(5)
memory usage: 418.3 MB


In [60]:
df19.columns = df19.columns.str.lower().str.replace(r"\s+", "_", regex=True)
df19.columns

Index(['trip_id', 'trip_duration', 'start_station_id', 'start_time',
       'start_station_name', 'end_station_id', 'end_time', 'end_station_name',
       'bike_id', 'user_type'],
      dtype='str')

In [ ]:
df19.isna().sum() # nothing to restore

trip_id                 0
trip_duration          16
start_station_id        0
start_time              0
start_station_name      0
end_station_id        454
end_time                0
end_station_name      454
bike_id                 0
user_type               0
dtype: int64

In [61]:
st19 = df19[['start_station_id', 'start_station_name']]

In [338]:
df19['start_station_name'] = [normalize_station_name(name) for name in df19.start_station_name]
df19['end_station_name'] = [normalize_station_name(name) for name in df19.end_station_name]

In [339]:
print('before: ', df19.memory_usage(deep=True).sum()/1e9)
df19 = optimize_df(df19)
print('after: ', df19.memory_usage(deep=True).sum()/1e9)

before:  0.438224907
after:  0.090295838


In [341]:
df19.to_parquet('clean_years/df19.parquet', index=False)

In [62]:
del df19
gc.collect()

6

### 2018


In [63]:
path18 = 'raw_data/bikeshare2018'
quaterly_dfs_18 = [pd.read_csv(os.path.join(path18, file)) for file in os.listdir(path18)]

In [64]:
df18 = pd.concat(quaterly_dfs_18, ignore_index=True)
df18.head()

,trip_id,trip_duration_seconds,from_station_id,trip_start_time,from_station_name,trip_stop_time,to_station_id,to_station_name,user_type
0,3244396,528,7254,7/1/2018 0:00,Borden St / Bloor St W - SMART,7/1/2018 0:08,7140,Macpherson Ave / Spadina Rd,Annual Member
1,3244398,570,7031,7/1/2018 0:00,Jarvis St / Isabella St,7/1/2018 0:09,7167,Queen St E / George St (Moss Park),Annual Member
2,3244397,368,7286,7/1/2018 0:00,Gerrard St E / Broadview - SMART,7/1/2018 0:06,7286,Gerrard St E / Broadview - SMART,Casual Member
3,3244399,1147,7248,7/1/2018 0:00,Baldwin Ave / Spadina Ave - SMART,7/1/2018 0:19,7020,Phoebe St / Spadina Ave,Annual Member
4,3244400,730,7207,7/1/2018 0:02,Dundas St W / Crawford St,7/1/2018 0:14,7000,Fort York Blvd / Capreol Ct,Annual Member


In [345]:
df18.info()

<class 'pandas.DataFrame'>
RangeIndex: 1922955 entries, 0 to 1922954
Data columns (total 9 columns):
 #   Column                 Dtype
---  ------                 -----
 0   trip_id                int64
 1   trip_duration_seconds  int64
 2   from_station_id        int64
 3   trip_start_time        str  
 4   from_station_name      str  
 5   trip_stop_time         str  
 6   to_station_id          int64
 7   to_station_name        str  
 8   user_type              str  
dtypes: int64(4), str(5)
memory usage: 309.0 MB


In [65]:
df18.columns = df18.columns.str.lower().str.replace(r'\s+', '_', regex=True)
df18.columns

Index(['trip_id', 'trip_duration_seconds', 'from_station_id',
       'trip_start_time', 'from_station_name', 'trip_stop_time',
       'to_station_id', 'to_station_name', 'user_type'],
      dtype='str')

In [66]:
df18 = df18.rename(columns={'trip_duration_seconds':'trip_duration', 
                        'from_station_id':'start_station_id',
                        'trip_start_time':'start_time', 
                        'from_station_name':'start_station_name', 
                        'to_station_id':'end_station_id', 
                        'to_station_name':'end_station_name', 
                        'trip_stop_time':'end_time'})

In [350]:
df18.isna().sum()

trip_id               0
trip_duration         0
start_station_id      0
start_time            0
start_station_name    0
end_time              0
end_station_id        0
end_station_name      0
user_type             0
dtype: int64

In [67]:
st18 = df18[['start_station_id', 'start_station_name']]

In [351]:
df18['start_station_name'] = [normalize_station_name(name) for name in df18.start_station_name]
df18['end_station_name'] = [normalize_station_name(name) for name in df18.end_station_name]

In [358]:
# update sations names:
(df18.merge(stations_actual_names, 
           how='left',
           left_on = 'start_station_id', 
           right_on='station_id')
       # .query('name.isna()')
    .query('start_station_name!=name')[['name', 'start_station_name']]
    .drop_duplicates()[:4]
    )
# 89 changes since 2018

,name,start_station_name
2,Gerrard St E / Broadview Ave - SMART,Gerrard St E / Broadview - SMART
3,Baldwin St / Spadina Ave - SMART,Baldwin Ave / Spadina Ave - SMART
11,Atlantic Ave / Liberty St,Exhibition GO (Atlantic Ave)
12,Yonge St / Lake Shore Blvd E,Yonge St / Harbour St


In [359]:
df18 = (df18.merge(stations_actual_names, 
           how='left',
           left_on = 'start_station_id', 
           right_on='station_id')
        # assign newest version, fill if there are any NAs
    .assign(start_station_name = lambda d: d['name'].combine_first(d['start_station_name']))
    .drop(['name', 'station_id'], axis=1)
    )

In [ ]:
# check the result:
(df18.merge(stations_actual_names, 
           how='left',
           left_on = 'start_station_id', 
           right_on='station_id')
    .query('start_station_name!=name')[['name', 'start_station_name']]
    .drop_duplicates()[:4]
    )

,name,start_station_name


In [361]:
# memory optimization:
print('before: ', df18.memory_usage(deep=True).sum()/1e9)
df18 = optimize_df(df18)
print('after: ', df18.memory_usage(deep=True).sum()/1e9)

before:  0.323564967
after:  0.063482804


In [363]:
df18.info()

<class 'pandas.DataFrame'>
RangeIndex: 1922955 entries, 0 to 1922954
Data columns (total 9 columns):
 #   Column              Dtype         
---  ------              -----         
 0   trip_id             int32         
 1   trip_duration       int32         
 2   start_station_id    int16         
 3   start_time          datetime64[us]
 4   start_station_name  category      
 5   end_time            datetime64[us]
 6   end_station_id      int16         
 7   end_station_name    category      
 8   user_type           category      
dtypes: category(3), datetime64[us](2), int16(2), int32(2)
memory usage: 60.5 MB


In [364]:
df18.to_parquet('clean_years/df18.parquet', index=False)

In [581]:
del df18
gc.collect()

6

### 2017

In [68]:
# each quarter differs with date format. 
path17 = 'raw_data/2017 Data'

In [69]:
def parse_dt(s):
    first = pd.to_numeric(s.dropna().astype(str).str.split('/').str[0], errors='coerce')
    return pd.to_datetime(s, dayfirst=bool((first > 12).any()), format='mixed', errors='coerce')


quaterly_dfs_17 = []
for file in os.listdir(path17):
    tmp = pd.read_csv(os.path.join(path17, file))
    tmp['trip_start_time'] = parse_dt(tmp['trip_start_time'])
    tmp['trip_stop_time'] = parse_dt(tmp['trip_stop_time'])
    quaterly_dfs_17.append(tmp)

In [70]:
df17 = pd.concat(quaterly_dfs_17, ignore_index=True)
df17.head()

,trip_id,trip_start_time,trip_stop_time,trip_duration_seconds,from_station_id,from_station_name,to_station_id,to_station_name,user_type
0,868665,2017-04-01 00:01:00,2017-04-01 00:06:00,297,7073.0,Bremner Blvd / Spadina Ave,7211.0,Fort York Blvd / Garrison Rd,Member
1,868666,2017-04-01 00:07:00,2017-04-01 00:11:00,236,7038.0,Dundas St / Yonge St,7163.0,Yonge St / Wood St,Member
2,868667,2017-04-01 00:08:00,2017-04-01 00:14:00,380,7012.0,Elizabeth St / Edward St (Bus Terminal),7069.0,Queen St W / Spadina Ave,Member
3,868668,2017-04-01 00:12:00,2017-04-01 00:34:00,1299,7172.0,Strachan Ave / Princes' Blvd,7069.0,Queen St W / Spadina Ave,Member
4,868669,2017-04-01 00:13:00,2017-04-01 00:18:00,323,7039.0,Simcoe St / Dundas St W,7021.0,Bay St / Albert St,Member


In [370]:
df17.info()

<class 'pandas.DataFrame'>
RangeIndex: 1492369 entries, 0 to 1492368
Data columns (total 9 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   trip_id                1492369 non-null  int64         
 1   trip_start_time        1492369 non-null  datetime64[us]
 2   trip_stop_time         1492368 non-null  datetime64[us]
 3   trip_duration_seconds  1492369 non-null  int64         
 4   from_station_id        465476 non-null   float64       
 5   from_station_name      1492369 non-null  str           
 6   to_station_id          465476 non-null   float64       
 7   to_station_name        1492368 non-null  str           
 8   user_type              1492369 non-null  str           
dtypes: datetime64[us](2), float64(2), int64(2), str(3)
memory usage: 185.6 MB


In [71]:
df17.columns = df17.columns.str.lower().str.replace(r"\s+", "_", regex=True)
df17.columns

Index(['trip_id', 'trip_start_time', 'trip_stop_time', 'trip_duration_seconds',
       'from_station_id', 'from_station_name', 'to_station_id',
       'to_station_name', 'user_type'],
      dtype='str')

In [72]:
df17 = df17.rename(columns={'trip_duration_seconds':'trip_duration', 
                        'from_station_id':'start_station_id',
                        'trip_start_time':'start_time', 
                        'from_station_name':'start_station_name', 
                        'to_station_id':'end_station_id', 
                        'to_station_name':'end_station_name', 
                        'trip_stop_time':'end_time'})

In [373]:
df17.isna().sum()

trip_id                     0
start_time                  0
end_time                    1
trip_duration               0
start_station_id      1026893
start_station_name          0
end_station_id        1026893
end_station_name            1
user_type                   0
dtype: int64

In [73]:
st17 = df17[['start_station_id', 'start_station_name']]

In [374]:
df17['start_station_name'] = [normalize_station_name(name) for name in df17.start_station_name]
df17['end_station_name'] = [normalize_station_name(name) for name in df17.end_station_name]

In [57]:
df17.isna().sum()

trip_id                     0
start_time                  0
end_time                    1
trip_duration               0
start_station_id      1026893
start_station_name          0
end_station_id        1026893
end_station_name            1
user_type                   0
dtype: int64

#### Information recovery for start_station_id/end_station_name 2017

In [378]:
df17 = (df17.merge(stations_vocab, 
           how='left', 
           left_on = 'start_station_name',
           right_on='name'
           )
    .assign(start_station_id = lambda d: d['start_station_id'].combine_first(d['station_id']))
    .drop(['name', 'station_id'], axis=1)
    
    .merge(stations_vocab, 
           how='left', 
           left_on = 'end_station_name',
           right_on='name'
           )
    .assign(end_station_id = lambda d: d['end_station_id'].combine_first(d['station_id']))
    .drop(['name', 'station_id'], axis=1)
    # .isna().sum()
 )

In [422]:
df17[df17.start_station_id.isna()]['start_station_name'].unique()

<ArrowStringArray>
[            'Michael Sweet Ave / St Patrick St',
                       'Beverly St / College St',
                   'College St / Margueretta St',
                      'Lansdowne Subway Green P',
                        'College St / Roxton Rd',
                                  'Base Station',
 'Lake Shore Blvd W / Ontario Dr(Ontario Place)',
           'Dovercourt Rd / Harrison St - SMART',
        'MacLennan Ave - SMART / Summerhill Ave',
                      'Fringe Next Stage - 7219']
Length: 10, dtype: str

In [406]:
# df17[df17.end_station_id.isna()]['end_station_name'].unique()

In [485]:
stations_actual_names[stations_actual_names.name.str.contains('Lake Shore')][:5]

,station_id,name,is_smart,name_normalized
3,7259,Lake Shore Blvd W / Lower Spadina Ave,0,Lake Shore Blvd W / Lower Spadina Ave
14,7398,Lake Shore Blvd W / York St,0,Lake Shore Blvd W / York St
150,7171,Lake Shore Blvd W / Ontario Place Blvd,0,Lake Shore Blvd W / Ontario Place Blvd
317,7242,Lake Shore Blvd W / Ontario Dr,0,Lake Shore Blvd W / Ontario Dr
344,7220,Ellis Ave / Lake Shore Blvd W,0,Ellis Ave / Lake Shore Blvd W


In [ ]:
# Lake Shore Blvd W / Ontario Dr(Ontario Place)
# Ontario Place Blvd / Lake Shore Blvd W OR Lake Shore Blvd W / Ontario Dr ????

In [ ]:
name = 'Lake Shore Blvd W / Ontario Dr(Ontario Place)'
parts = sorted([p.strip().replace(r"\s+", " ") for p in str(name).strip().split('/')])
parts
# how to deal with (Ontario Place)?

['Lake Shore Blvd W', 'Ontario Dr(Ontario Place)']

### Clean stations vocabularies: 

In [418]:
def standardize_intersection(name):
    if pd.isna(name):
        return name
    
    name = str(name).strip()
    
    # if name contains /
    if '/' in name:
        # split by /, strip spaces, sort alfabeticaly, rejoin
        parts = sorted([p.strip().replace(r"\s+", " ") for p in name.split('/')])
        return ' / '.join(parts)
    return name


In [419]:
stations_actual_names['name'] = [standardize_intersection(name) for name in stations_actual_names['name']]

In [445]:
stations_actual_names['is_smart'] = stations_actual_names['name'].str.lower().str.contains('smart').astype('int16')

In [452]:
# remove suffix SMART
stations_actual_names.name = [re.sub(r'\s*[-–]\s*SMART\b', '', name, flags=re.IGNORECASE) for name in stations_actual_names.name]

In [454]:
df17['start_station_name'] = [re.sub(r'\s*[-–]\s*SMART\b', '', name, flags=re.IGNORECASE) for name in df17.start_station_name]
# df17['end_station_name'] = [re.sub(r'\s*[-–]\s*SMART\b', '', name, flags=re.IGNORECASE) for name in df17.end_station_name]


In [ ]:
#  once more after removing suffix SMART
stations_actual_names['name'] = [standardize_intersection(name) for name in stations_actual_names['name']]
df17['start_station_name'] = [standardize_intersection(name) for name in df17['start_station_name']]

In [ ]:
# check matching after all standartizations:
(df17[df17['start_station_id'].isna()][['start_station_name']]
    .drop_duplicates()
    .merge(stations_actual_names, 
           left_on='start_station_name', 
           right_on='name')
    )
# still nothing

,start_station_name,station_id,name,is_smart


### Fuzzy matching

In [402]:
from rapidfuzz import process, fuzz

In [426]:
unmatched_names = df17[df17['start_station_id'].isna()]['start_station_name'].dropna().unique()
valid_names = stations_actual_names['name'].tolist()

In [487]:
matches = {}
for name in unmatched_names:
    best_match, score, _ = process.extractOne(name, valid_names, scorer=fuzz.ratio)
    # if score > 85:  # High threshold to ensure accuracy
    matches[name] = [best_match, np.round(score, 1)]


matches

{'Michael Sweet Ave / St Patrick St': ['Michael Sweet Ave / Simcoe St',
  np.float64(83.9)],
 'Beverly St / College St': ['Beverley St / College St', np.float64(97.9)],
 'College St / Margueretta St': ['College St / Markham St', np.float64(80.0)],
 'Lansdowne Subway Green P': ['Pape Subway Green P', np.float64(79.1)],
 'College St / Roxton Rd': ['College St / Huron St', np.float64(74.4)],
 'Base Station': ['Wellesley Station', np.float64(69.0)],
 'Lake Shore Blvd W / Ontario Dr(Ontario Place)': ['Lake Shore Blvd W / Ontario Dr',
  np.float64(80.0)],
 'Dovercourt Rd / Harrison St': ['Dovercourt Rd / Queen St W',
  np.float64(75.5)],
 'MacLennan Ave / Summerhill Ave': ['Maclennan Ave / Summerhill Ave',
  np.float64(96.7)],
 'Fringe Next Stage - 7219': ['Front St E / Scott St (2)', np.float64(44.9)]}

In [ ]:
# let's check the importance of information on parenthesis:
stations_actual_names[stations_actual_names.name.str.contains(r"\(", regex=True)]

,station_id,name,is_smart
1,8001,Union Station (North),1
13,7014,Carlton St (Allan Gardens) / Sherbourne St,0
19,7175,HTO Park (Queens Quay W),0
43,7048,Front St W / Yonge St (Hockey Hall of Fame),0
45,7164,Gould St / Yonge St (TMU),0
...,...,...,...
1297,7167,George St (Moss Park) / Queen St E,0
1299,7634,Gerrard St W (WEST) / University Ave,1
1359,7426,Fleet St / Garrison Rd (Fort York),0
1360,7179,Bloor GO / UP Station (West Toronto Railpath),0


In [484]:
stations_actual_names['name_normalized'] = stations_actual_names['name'].str.replace(r'\s*\(.*?\)', '', regex=True)

In [60]:
(df17
    .groupby('start_station_name')['trip_id']
    .nunique()
    .reset_index(name='nr_trips')
    .sort_values('nr_trips')
    .head()
    )

,start_station_name,nr_trips
12,Base Station,2
111,Fringe Next Stage - 7219,36
267,Victoria Park Subway Station - SMART,165
98,Dundas St W / Roncesvalles Green P - SMART,187
249,Summerhill Ave / MacLennan Ave - SMART,201


In [114]:
pd.set_option('display.max_rows', 20)

In [115]:
(pd.DataFrame([x.split('/') for x in df17.start_station_name.unique()])
        .rename(columns={0: 'street_1', 1: 'street_2'})
        .sort_values(['street_1', 'street_2'])
        )

/var/folders/xf/ygl5_zk57kn1rh2h1xj9fyx40000gn/T/ipykernel_96582/1565718847.py:1: Pandas4Warning: Constructing a DataFrame from a list of sequences with mismatched lengths is deprecated and will raise in a future version. The shorter sequences are currently padded with NaN; make all sequences the same length before constructing instead.
  (pd.DataFrame([x.split('/') for x in df17.start_station_name.unique()])


,street_1,street_2,2
264,111 Bond St (North of Dundas St E) - SMART,NaN,NaN
22,161 Bleecker St (South of Wellesley),NaN,NaN
120,25 York St (ACC,Union Station South),NaN
108,424 Wellington St W,NaN,NaN
85,51 Parliament St,NaN,NaN
...,...,...,...
139,Yonge St,Harbour St,NaN
123,Yonge St,Wood St,NaN
118,Yonge St,Yorkville Ave,NaN
282,York St,King St W - SMART,NaN


In [ ]:
df17.start_station_name.nunique() #from 293 to 290

290

In [116]:
(df17
    .groupby('start_station_name')['trip_id']
    .nunique()
    .reset_index(name='nr_trips')
    .sort_values('nr_trips')
    .head()
    )

,start_station_name,nr_trips
11,Base Station,2
110,Fringe Next Stage - 7219,36
264,Victoria Park Subway Station - SMART,165
97,Dundas St W / Roncesvalles Green P - SMART,187
246,Summerhill Ave / MacLennan Ave - SMART,201


In [124]:
# check Base Station
df17[df17.start_station_name.isin(['Base Station'])]

,trip_id,start_time,end_time,trip_duration,start_station_id,start_station_name,end_station_id,end_station_name,user_type
599081,1542474,2017-08-09 10:44:00,2017-08-09 14:16:00,12684,NaN,Base Station,NaN,Queen St W / Gladstone Ave,Member
1303397,2314950,2017-12-01 14:43:54,2017-12-01 14:44:02,8,NaN,Base Station,NaN,Base Station,Member


In [126]:
# 2 strange trips, the first one lasted more then 3 days, the second one - 8 sec. Looks like test - noisy data, better to remove
df17 = df17[~df17.start_station_name.isin(['Base Station'])]

In [128]:
# the problem with 2017 - lack of ids
# let's use parsed data from stations.parquet from the geodata step
stations = pd.read_parquet('stations.parquet')


In [144]:
stations['name'] = [normalize_station_name(name) for name in stations.name]

In [145]:
stations.head()

,station_id,name,lat,lon
0,7254,Borden St / Bloor St W - SMART,43.665461,-79.408480
1,7402,Wellington St W / Bathurst St,43.642624,-79.402459
2,7456,Belmont St / Davenport Rd - SMART,43.674381,-79.392489
3,7470,York St / Lake Shore Blvd W,43.641960,-79.380806
4,7481,Westmount Ave / St Clair Ave W - SMART,43.678510,-79.442150


In [147]:
dup_names = stations[stations.duplicated(subset='name', keep=False)].sort_values('name')
dup_names


,station_id,name,lat,lon
7,7051,Mutual St / Shuter St,43.654879,-79.375091
667,7693,Mutual St / Shuter St,43.654920,-79.374961
3,7470,York St / Lake Shore Blvd W,43.641960,-79.380806
437,7398,York St / Lake Shore Blvd W,43.641952,-79.380723


In [148]:
station_name_to_id = stations.drop_duplicates(subset='name', keep='last').set_index('name')['station_id']
df17['start_station_id'] = df17['start_station_name'].map(station_name_to_id)

print(df17['start_station_id'].notna().sum())
print(len(df17))


1078044
1492367


In [154]:
# df17.dtypes

In [152]:
pd.DataFrame(df17[df17.start_station_id.isna()]['start_station_name'].unique())

,0
0,Bremner Blvd / Spadina Ave
1,Dundas St / Yonge St
2,Front St / Yonge St (Hockey Hall of Fame)
3,Beverly St / Dundas St W
4,College St W / Huron St
...,...
87,York St / King St W - SMART
88,Dovercourt Rd / Harrison St - SMART
89,Summerhill Ave / MacLennan Ave - SMART
90,Bloor St W / Shaw Ave - SMART


### 2016

In [584]:
import openpyxl

In [586]:
df16 = pd.read_excel('raw_data/bikeshare-ridership-2016.xlsx')

In [587]:
df16.isna().sum()

trip_id                  0
trip_start_time          0
trip_stop_time           0
trip_duration_seconds    0
from_station_name        3
to_station_name          2
user_type                0
dtype: int64

In [589]:
df16.columns = [c.lower().replace(' ', '_') for c in df16.columns]
df16.columns

Index(['trip_id', 'trip_start_time', 'trip_stop_time', 'trip_duration_seconds',
       'from_station_name', 'to_station_name', 'user_type'],
      dtype='str')

In [590]:
df16 = df16.rename(columns={'trip_duration_seconds':'trip_duration', 
                        'from_station_id':'start_station_id',
                        'trip_start_time':'start_time', 
                        'from_station_name':'start_station_name', 
                        'to_station_id':'end_station_id', 
                        'to_station_name':'end_station_name', 
                        'trip_stop_time':'end_time'})

In [ ]:
# st16 = df16[['start_station_id', 'start_station_name']]

KeyError: "['start_station_id'] not in index"

In [63]:
df16 = optimize_df(df16)
df16.memory_usage(deep=True).sum()/1e9

np.float64(0.009948705)

In [67]:
df16.info()

<class 'pandas.DataFrame'>
RangeIndex: 367962 entries, 0 to 367961
Data columns (total 7 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   trip_id             367962 non-null  int32         
 1   start_time          367962 non-null  datetime64[us]
 2   end_time            367962 non-null  datetime64[us]
 3   trip_duration       367962 non-null  int16         
 4   start_station_name  367959 non-null  category      
 5   end_station_name    367960 non-null  category      
 6   user_type           367962 non-null  category      
dtypes: category(3), datetime64[us](2), int16(1), int32(1)
memory usage: 9.5 MB


In [68]:
df16.start_time.dt.year.unique()

array([2016, 2000], dtype=int32)

In [69]:
df16[df16.start_time.dt.year==2000]

,trip_id,start_time,end_time,trip_duration,start_station_name,end_station_name,user_type
365004,622515,2000-01-01 14:40:00,2000-01-01 15:44:00,3794,Front St / Yonge St (Hockey Hall of Fame),Front St / Yonge St (Hockey Hall of Fame),Member


In [ ]:
df16[df16.trip_id<622515].sort_values('trip_id', ascending=False)

,trip_id,start_time,end_time,trip_duration,start_station_name,end_station_name,user_type
91687,462304,2016-09-30 23:59:00,2016-10-01 00:12:00,771,80 Clinton St (North of College),424 Wellington St. W,Member
361055,462303,2016-09-30 23:59:00,2016-10-01 00:04:00,317,Wellesley St E / Yonge St Green P,Hayter St / Laplante Ave,Member
48925,462302,2016-09-30 23:59:00,2016-10-01 00:04:00,341,Queen St W / Spadina Ave,Augusta Ave / Denison Sq,Member
270823,462301,2016-09-30 23:59:00,2016-10-01 00:13:00,880,Dockside Dr / Queens Quay E (Sugar Beach),Queen St E / Berkely St,Member
67715,462300,2016-09-30 23:57:00,2016-10-01 00:06:00,556,Dundas St E / Regent Park Blvd,King St E / Jarvis St,Member
...,...,...,...,...,...,...,...
128416,24012,2016-07-01 00:01:00,2016-07-01 00:12:00,662,Simcoe St / Queen St W,Queen St W / Spadina Ave,Member
169223,24011,2016-07-01 00:01:00,2016-07-01 00:07:00,399,Trinity St /Front St E,Princess St / Adelaide St,Member
172573,24010,2016-07-01 00:00:00,2016-07-01 00:42:00,2487,Bay St / Queens Quay W (Ferry Terminal),York St / Queens Quay W,Casual
110732,24009,2016-07-01 00:00:00,2016-07-01 00:10:00,603,Wellington St W / Bay St,King St W / Spadina Ave,Member


In [ ]:
df16.sort_values('start_time', ascending=False).head()

,trip_id,start_time,end_time,trip_duration,start_station_name,end_station_name,user_type
361055,462303,2016-09-30 23:59:00,2016-10-01 00:04:00,317,Wellesley St E / Yonge St Green P,Hayter St / Laplante Ave,Member
48925,462302,2016-09-30 23:59:00,2016-10-01 00:04:00,341,Queen St W / Spadina Ave,Augusta Ave / Denison Sq,Member
270823,462301,2016-09-30 23:59:00,2016-10-01 00:13:00,880,Dockside Dr / Queens Quay E (Sugar Beach),Queen St E / Berkely St,Member
91687,462304,2016-09-30 23:59:00,2016-10-01 00:12:00,771,80 Clinton St (North of College),424 Wellington St. W,Member
67715,462300,2016-09-30 23:57:00,2016-10-01 00:06:00,556,Dundas St E / Regent Park Blvd,King St E / Jarvis St,Member


In [70]:
df16 = df16[~(df16.start_time.dt.year==2000)]

In [71]:
df16.isna().sum()

trip_id               0
start_time            0
end_time              0
trip_duration         0
start_station_name    3
end_station_name      2
user_type             0
dtype: int64

In [ ]:
df16.head()

,trip_id,start_time,end_time,trip_duration,start_station_name,end_station_name,user_type
0,53279,2016-07-09 01:03:00,2016-07-09 01:15:00,714,Dundas St E / Regent Park Blvd,Danforth Ave / Ellerbeck St,Member
1,53394,2016-07-09 02:15:00,2016-07-09 02:22:00,417,Riverdale Park North (Broadview Ave),Dundas St E / Regent Park Blvd,Member
2,58314,2016-07-10 17:04:00,2016-07-10 17:36:00,1904,Dundas St E / Regent Park Blvd,Queen St W / Close Ave,Member
3,60784,2016-07-11 01:45:00,2016-07-11 01:58:00,784,Union Station,Dundas St E / Regent Park Blvd,Member
4,93164,2016-07-18 13:35:00,2016-07-18 13:42:00,443,Front St W / Blue Jays Way,Front St / Yonge St (Hockey Hall of Fame),Member


In [ ]:
(df16
    .groupby('start_station_name')['trip_id']
    .nunique()
    .reset_index(name='nr_trips')
    .sort_values('nr_trips')
    .head()
    )

,start_station_name,nr_trips
55,Danforth Ave / Westlake Ave,83
193,Woodbine Subway Green P (Cedarvale Ave),109
128,Ossington Ave / College St W,110
170,Stephenson Ave / Main St,143
50,Danforth Ave / Barrington Ave,153


In [ ]:
df16.start_station_name.nunique()

200

In [170]:
df16['start_station_name'] = [normalize_station_name(name) for name in df16.start_station_name]
df16['end_station_name'] = [normalize_station_name(name) for name in df16.end_station_name]

In [159]:
df16.isna().sum()

trip_id               0
start_time            0
end_time              0
trip_duration         0
start_station_name    3
end_station_name      2
user_type             0
dtype: int64

#### Information recovery for start/end_station_id 2016

In [ ]:
# let's use stations_vocab created in geodata.ipynb for assigning station_ids.
stations_vocab = pd.read_parquet('stations_vocab.parquet')
stations_vocab.head()

,station_id,name
0,7000,Ft York / Capreol Crt
1,7000,Fort York Blvd / Capreol Crt
2,7000,Fort York Blvd / Capreol Ct
3,7001,Lower Jarvis St / The Esplanade
4,7001,Wellesley Station Green P (EBS)


In [173]:
# manage to assign all ids except where start_station_name isna
df16 = (df16
    .merge(stations_vocab, left_on='start_station_name', right_on='name', how='left')
    .drop(['name'], axis=1)
    .rename(columns={'station_id': 'start_station_id'})
    # do the same for end_station_name
    .merge(stations_vocab, left_on='end_station_name', right_on='name', how='left')
    .rename(columns={'station_id': 'end_station_id'})
    .drop(['name'], axis=1)
)

In [ ]:
# now when i finally assigne ids to name (whatever spelling it has), i need to assigne the name that is from the most rescent vocab, 
# if the station still exists. it means take the whole df from all the years, drop_duplicates on id and keep the last name. 
# (calculated in geodata.ipynb)

In [176]:
stations_actual_names = pd.read_parquet('stations_actual_names.parquet')
stations_actual_names.head()

,station_id,name
0,7719,Wolseley St / Augusta Ave
1,8001,Union Station (North) - SMART
2,7172,Strachan Ave / Princes Blvd
3,7259,Lower Spadina Ave / Lake Shore Blvd W
4,7713,Temperance St / Bay St


In [183]:
# merge df16  with stations_actual_names for updating station names
df16 = (df16
    .merge(stations_actual_names, 
        left_on='start_station_id', 
        right_on='station_id',
        how='left')
    # .query('name.isna()') # no na for name except where start_station_name initially is na
    .drop(['start_station_name', 'station_id'], axis=1)
    # update start_station_name 
    .rename(columns={'name': 'start_station_name'})
    
    # do the same for end_station_name
    .merge(stations_actual_names, 
        left_on='end_station_id', 
        right_on='station_id',
        how='left')
    .drop(['end_station_name', 'station_id'], axis=1)
    .rename(columns={'name': 'end_station_name'})
       
 )

In [187]:
df16.memory_usage(deep=True).sum()/1e9

np.float64(0.041227039)

In [188]:
# final optimization:
# df16.info()

df16 = optimize_df(df16)
df16.memory_usage(deep=True).sum()/1e9

np.float64(0.013557319)

In [189]:
df16.info()

<class 'pandas.DataFrame'>
RangeIndex: 386952 entries, 0 to 386951
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   trip_id             386952 non-null  int32         
 1   start_time          386952 non-null  datetime64[us]
 2   end_time            386952 non-null  datetime64[us]
 3   trip_duration       386952 non-null  int16         
 4   user_type           386952 non-null  category      
 5   start_station_id    386949 non-null  float32       
 6   end_station_id      386950 non-null  float32       
 7   start_station_name  386949 non-null  category      
 8   end_station_name    386950 non-null  category      
dtypes: category(3), datetime64[us](2), float32(2), int16(1), int32(1)
memory usage: 12.9 MB


### 2014-2015

In [40]:
df14_15 = pd.read_excel('bikeshare-ridership-2014-2015.xlsx', header=1)
df14_15.tail()

,Start Terminal,End Terminal,Casual,Registered,Total
5578,7079,7076.0,NaN,1.0,1
5579,7079,7077.0,NaN,4.0,4
5580,7079,7078.0,NaN,4.0,4
5581,7079,7079.0,1.0,4.0,5
5582,Total,NaN,6039.0,51057.0,57096


In [41]:
df14_15.rename(columns={'trip_duration_seconds':'trip_duration', 
                        'from_station_id':'start_station_id',
                        'trip_start_time':'start_time', 
                        'from_station_name':'start_station_name', 
                        'to_station_id':'end_station_id', 
                        'to_station_name':'end_station_name', 
                        'trip_stop_time':'end_time'}, inplace=True)

/var/folders/xf/ygl5_zk57kn1rh2h1xj9fyx40000gn/T/ipykernel_96582/1129840674.py:1: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df14_15.rename(columns={'trip_duration_seconds':'trip_duration',


In [42]:
df14_15.head()

,Start Terminal,End Terminal,Casual,Registered,Total
0,7000,7000.0,7.0,24.0,31
1,7000,7001.0,2.0,1.0,3
2,7000,7002.0,NaN,3.0,3
3,7000,7003.0,NaN,5.0,5
4,7000,7004.0,NaN,13.0,13


In [48]:
df14_15.tail()

,start_terminal,end_terminal,casual,registered,total
5578,7079,7076.0,NaN,1.0,1
5579,7079,7077.0,NaN,4.0,4
5580,7079,7078.0,NaN,4.0,4
5581,7079,7079.0,1.0,4.0,5
5582,Total,NaN,6039.0,51057.0,57096


In [44]:
df14_15.columns = [c.replace(' ', '_').lower() for c in df14_15.columns]

In [46]:
df14_15.start_terminal.nunique()

80

In [47]:
df14_15.end_terminal.nunique()

79

In [ ]:
# 57096 trips in Total, 80 Stations in 2015

### concatenate and save

In [131]:
df = pd.concat([df26, df25, df24, df21_23, df20, df19, df18, df17, df16])

In [7]:
def optimize_df(df):
    if 'start_time' in df.columns:
        df['start_time'] = pd.to_datetime(df['start_time'])
    if 'end_time' in df.columns:
        df['end_time'] = pd.to_datetime(df['end_time'])

    for col in ['bike_model', 'user_type', 'start_station_name', 'end_station_name']:
        if col in df.columns:
            df[col] = df[col].astype('category')

    for col in ['trip_id', 'trip_duration', 'start_station_id', 'end_station_id', 'bike_id']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    int_cols = df.select_dtypes(include='int64').columns
    df[int_cols] = df[int_cols].apply(pd.to_numeric, downcast='integer')
    float_cols = df.select_dtypes(include='float64').columns
    df[float_cols] = df[float_cols].apply(pd.to_numeric, downcast='float')
    return df

In [133]:
df = optimize_df(df)

In [134]:
df.dtypes

trip_id                        int64
trip_duration                float32
start_station_id             float32
start_time            datetime64[us]
start_station_name          category
end_station_id               float32
end_time              datetime64[us]
end_station_name            category
bike_id                      float32
user_type                   category
bike_model                  category
dtype: object

In [135]:
df.memory_usage(deep=True).sum()/1e9

np.float64(2.41059854)

In [117]:
df.columns

Index(['trip_id', 'trip_duration', 'start_station_id', 'start_time',
       'start_station_name', 'end_station_id', 'end_time', 'end_station_name',
       'bike_id', 'user_type', 'bike_model', 'trip__duration'],
      dtype='str')

In [137]:
del df26, df25, df24, df21_23, df20, df19, df18, df17, df16

In [138]:
gc.collect()

462

In [136]:
df.to_parquet('bikeshare_all.parquet', index=False)

## Information recovery for trip_duration

In [2]:
df = pd.read_parquet('bikeshare_all.parquet')

In [3]:
df['trip_duration_calc'] = (df['end_time'] - df['start_time']).dt.total_seconds()


In [4]:
df.loc[(df['trip_duration']==0) & (df['trip_duration_calc']!=0) & ~df['trip_duration_calc'].isna() & (df['trip_duration_calc']>0)].shape

(32, 12)

In [5]:
print(df['trip_duration'].isna().sum())
print(df[df['trip_duration']==0].shape[0])

265
44570


In [6]:
df[(df['trip_duration']==0) & (df['start_time']==df['end_time'])].shape[0]

29558

In [7]:
# i think we can skip all the trips above where trip_duration==0 and start/end time the same
mask = (df['trip_duration']==0) & (df['start_time']==df['end_time'])
df = df[~mask]

In [9]:
print(df['trip_duration'].isna().sum())
print(df[df['trip_duration']==0].shape[0])

265
15012


In [ ]:
# fill trip_duration (==0) with trip_duration_calc (end_time-start_time)
mask = (df['trip_duration']==0) & (df['trip_duration_calc']>0)
df.loc[mask, 'trip_duration'] = df.loc[mask, 'trip_duration_calc']

In [11]:
print(df['trip_duration'].isna().sum())
print(df[df['trip_duration']==0].shape[0])

265
14980


In [12]:
df[df['trip_duration']==0].start_time.dt.year.value_counts()

start_time
2026    8065
2024    5822
2025    1092
2017       1
Name: count, dtype: int64

In [13]:
df[(df['trip_duration']==0) & (df.start_time.dt.year==2017)]
# we can remove smth like that, it's not informative 

,trip_id,trip_duration,start_station_id,start_time,start_station_name,end_station_id,end_time,end_station_name,bike_id,user_type,bike_model,trip_duration_calc
44070839,2302635,0.0,NaN,2017-11-29 05:53:54,Seaton St / Dundas St E,NaN,NaT,NaN,NaN,Casual,NaN,NaN


In [185]:
df[(df['trip_duration']==0) & (df.start_time.dt.year==2026)].head()
# in this case there are plenty of information, let's dive deeper

,trip_id,trip_duration,start_station_id,start_time,start_station_name,end_station_id,end_time,end_station_name,bike_id,user_type,bike_model,trip_duration_calc
3821,43687212,0.0,7240.0,2026-01-02 14:38:08,Bloor St W / Shaw St - SMART,NaN,NaT,NaN,289.0,Member,ICONIC,NaN
6965,43691231,0.0,7322.0,2026-01-03 11:30:50,East Liberty St / Western Battery Rd,8023.0,NaT,Shaw St / College St,6062.0,Member,ICONIC,NaN
7364,43691833,0.0,7150.0,2026-01-03 13:05:01,Dufferin St / Sylvan Av (Dufferin Grove Park),8023.0,NaT,Shaw St / College St,5825.0,Member,ICONIC,NaN
18005,43705519,0.0,8180.0,2026-01-06 08:17:55,Osler Playground (Argyle St),NaN,NaT,NaN,3083.0,Casual,ICONIC,NaN
19260,43706967,0.0,7933.0,2026-01-06 10:05:57,Elizabeth St / Elm St,NaN,NaT,NaN,4354.0,Casual,ICONIC,NaN


In [14]:
# check where user arrived to the other station:
df[(df['trip_duration']==0) & (df['start_station_id']!=df['end_station_id'])].shape[0]

14559

In [197]:
df.shape[0] - 14559

44594574

In [15]:
# can remove with trip_duration==0 and start/end station the same (421 obs)
mask = (df['trip_duration']==0) & (df['start_station_id']==df['end_station_id'])
df = df.loc[~mask]

In [16]:
print(df['trip_duration'].isna().sum())
print(df[df['trip_duration']==0].shape[0])

265
14559


In [ ]:
# get rid of obs. where we do not know start_station_id and trip_duration==0
mask = (df['trip_duration']==0) & (df['start_station_id'].isna())
df = df[~mask]

In [28]:
df[(df['trip_duration']==0)]['end_station_id'].notna().mean()

np.float64(0.3872097815634016)

In [ ]:
# about 40% trips with trip_duration=0 but at the same time end_station_id!=start_station_id
# so it can't be zero duration

In [33]:
(df[(df['trip_duration']==0) & (df['end_station_id'].notna())]
        .sort_values(['start_station_id', 'start_time'])
        .head()) # 5637

,trip_id,trip_duration,start_station_id,start_time,start_station_name,end_station_id,end_time,end_station_name,bike_id,user_type,bike_model,trip_duration_calc
17536898,30003946,0.0,7000.0,2024-07-09 22:40:55,Fort York Blvd / Capreol Ct,7762.0,NaT,College St / Henry St,1178.0,Member,ICONIC,NaN
17536903,30003951,0.0,7000.0,2024-07-09 22:41:06,Fort York Blvd / Capreol Ct,7762.0,NaT,College St / Henry St,3466.0,Member,ICONIC,NaN
17643206,30127130,0.0,7000.0,2024-07-14 01:48:06,Fort York Blvd / Capreol Ct,7768.0,NaT,Armoury St / Chestnut St,3692.0,Member,ICONIC,NaN
17920399,30446161,0.0,7000.0,2024-07-23 15:38:44,Fort York Blvd / Capreol Ct,7322.0,NaT,East Liberty St / Western Battery Rd,2847.0,Casual,ICONIC,NaN
17964633,30496028,0.0,7000.0,2024-07-24 21:00:32,Fort York Blvd / Capreol Ct,7176.0,NaT,Bathurst St / Fort York Blvd,646.0,Member,ICONIC,NaN


In [34]:
(df[(df['trip_duration']==0) & (df['end_station_id'].isna())]
        .sort_values(['start_station_id', 'start_time'])
        .head())

,trip_id,trip_duration,start_station_id,start_time,start_station_name,end_station_id,end_time,end_station_name,bike_id,user_type,bike_model,trip_duration_calc
18274498,30854816,0.0,7000.0,2024-08-03 23:04:30,Fort York Blvd / Capreol Ct,NaN,NaT,NaN,2824.0,Casual,ICONIC,NaN
18340178,30932697,0.0,7000.0,2024-08-06 19:31:03,Fort York Blvd / Capreol Ct,NaN,NaT,NaN,8184.0,Casual,EFIT G5,NaN
18340235,30932759,0.0,7000.0,2024-08-06 19:31:57,Fort York Blvd / Capreol Ct,NaN,NaT,NaN,4129.0,Casual,ICONIC,NaN
18366378,30963114,0.0,7000.0,2024-08-07 17:37:43,Fort York Blvd / Capreol Ct,NaN,NaT,NaN,8105.0,Casual,EFIT G5,NaN
18379670,30977829,0.0,7000.0,2024-08-07 23:57:12,Fort York Blvd / Capreol Ct,NaN,NaT,NaN,7329.0,Casual,ICONIC,NaN


In [38]:
# i can assume it's invalid trips, and try to remove them for the further analysis. They doesn't look like a real trip, rather uder's try
mask = (df['trip_duration']==0) & (df['end_station_id'].isna())
df = df[~mask]

In [39]:
df[df['trip_duration']==0].start_time.dt.year.value_counts()

start_time
2026    3132
2024    2331
2025     174
Name: count, dtype: int64

In [40]:
df[df['trip_duration'].isna()].start_time.dt.year.value_counts()

start_time
2020    249
2019     16
Name: count, dtype: int64

In [42]:
# let's coalesce trip_duration with trip_duration_calc (end_time-start_time)
df['trip_duration'] = df['trip_duration'].combine_first(df['trip_duration_calc'])

In [43]:
df[df['trip_duration'].isna()].start_time.dt.year.value_counts()

Series([], Name: count, dtype: int64)

In [44]:
df[df['trip_duration']==0].start_time.dt.year.value_counts()

start_time
2026    3132
2024    2331
2025     174
2019      14
Name: count, dtype: int64

In [48]:
df[(df['trip_duration']==0) & (df.start_time.dt.year==2019)].head(3)

# after combining 14 observations appear in 2019. let's remove where end_station_id = start_station_id

,trip_id,trip_duration,start_station_id,start_time,start_station_name,end_station_id,end_time,end_station_name,bike_id,user_type,bike_model,trip_duration_calc
38865465,6034741,0.0,7064.0,2019-08-06 07:50:00,51 Parliament St,7100.0,2019-08-06 07:50:00,Dundas St E / Regent Park Blvd,2872.0,Annual Member,NaN,0.0
39250027,6465255,0.0,7417.0,2019-09-04 13:13:00,King St W / Jordan St,7184.0,2019-09-04 13:13:00,Ossington Ave / College St,3550.0,Annual Member,NaN,0.0
39324423,6547689,0.0,7324.0,2019-09-10 21:31:00,King St W / Charlotte St (West),7160.0,2019-09-10 21:31:00,King St W / Tecumseth St,4252.0,Annual Member,NaN,0.0


In [51]:
mask = (df['trip_duration']==0) & (df['start_station_id']==df['end_station_id'])
df = df[~mask]

In [52]:
df[df['trip_duration']==0].start_time.dt.year.value_counts()

start_time
2026    3132
2024    2331
2025     174
2019       5
Name: count, dtype: int64

In [ ]:
# I will proceede with this incomplete data and count the trips even with trip_duration==0, but where start and end station id are different
# my assumtion that there is a problem in data collection process. 
# but i am going to exclude the rows with trip_duration==0 for counting time statistics

In [ ]:
# remove supportinf column 
df.drop(columns=['trip_duration_calc'], inplace=True)

/var/folders/xf/ygl5_zk57kn1rh2h1xj9fyx40000gn/T/ipykernel_92657/800625305.py:2: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns=['trip_duration_calc'], inplace=True)


In [54]:
df.info()

<class 'pandas.DataFrame'>
Index: 44599781 entries, 0 to 44638690
Data columns (total 11 columns):
 #   Column              Dtype         
---  ------              -----         
 0   trip_id             int64         
 1   trip_duration       float64       
 2   start_station_id    float32       
 3   start_time          datetime64[us]
 4   start_station_name  category      
 5   end_station_id      float32       
 6   end_time            datetime64[us]
 7   end_station_name    category      
 8   bike_id             float32       
 9   user_type           category      
 10  bike_model          category      
dtypes: category(4), datetime64[us](2), float32(3), float64(1), int64(1)
memory usage: 2.4 GB


In [ ]:
# df = df[df.start_time.dt.year!=2000]!!!!

In [ ]:
# df.to_parquet('bikeshare_clean.parquet', index=False) 

In [594]:
# del df20, df17, df19

In [91]:
stations = pd.concat([st17, st18, st19, st20, st21, st22, st23, st24, st25, st26], ignore_index=True).drop_duplicates()

In [92]:
stations.isna().sum()

start_station_id      272
start_station_name    244
dtype: int64

In [100]:
stations = stations.dropna(how='all')

In [103]:
stations[stations.start_station_id.isna() | stations.start_station_name.isna()]

,start_station_id,start_station_name
333353,NaN,Princess St / Adelaide St E
333354,NaN,Fort York Blvd / Capreol Crt
333356,NaN,Elizabeth St / Edward St (Bus Terminal)
333358,NaN,Bay St / St. Joseph St
333359,NaN,Bay St / College St (East Side)
...,...,...
21956586,7882,NaN
21964822,7917,NaN
21966907,7916,NaN
21983573,7915,NaN


In [106]:
# pd.to_numeric(stations.start_station_id, errors='coerce')

In [107]:
stations.to_json("raw_stations.json")